In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("Pandas version:", pd.__version__)
print("Environment ready.")

Pandas version: 3.0.3
Environment ready.


In [3]:
DATA_DIR = Path(r"C:\Users\aduak\Downloads\capstone")

files = {
    "Calgary": DATA_DIR / "CAN_AB_CALGARY-INTL-A_3031092_CWEEDS2011_1998-2017.csv",
    "Edmonton": DATA_DIR / "CAN_AB_EDMONTON-INTL-A_3012216_CWEEDS2011_1998-2017.csv",
    "Lethbridge": DATA_DIR / "CAN_AB_LETHBRIDGE-CDA_3033890_CWEEDS2011_1998-2017.csv",
    "Medicine Hat": DATA_DIR / "CAN_AB_MEDICINE-HAT-RCS_3034485_CWEEDS2011_1998-2017.csv",
    "SAIT": DATA_DIR / "solar data gbtac.csv"
}

for name, path in files.items():
    print(f"{name:15} -> {path.exists()} -> {path.name}")

Calgary         -> True -> CAN_AB_CALGARY-INTL-A_3031092_CWEEDS2011_1998-2017.csv
Edmonton        -> True -> CAN_AB_EDMONTON-INTL-A_3012216_CWEEDS2011_1998-2017.csv
Lethbridge      -> True -> CAN_AB_LETHBRIDGE-CDA_3033890_CWEEDS2011_1998-2017.csv
Medicine Hat    -> True -> CAN_AB_MEDICINE-HAT-RCS_3034485_CWEEDS2011_1998-2017.csv
SAIT            -> True -> solar data gbtac.csv


In [4]:
# Inspect the first few raw lines of the Calgary CWEEDS file

with open(files["Calgary"], "r", encoding="utf-8-sig") as f:
    for i in range(5):
        print(f"LINE {i+1}:")
        print(f.readline())
        print("-" * 100)

LINE 1:
Data version,Climate station name,Province,Country,Station ID,Degree latitude,Degree longitude,UTC offset,Elevation

----------------------------------------------------------------------------------------------------
LINE 2:
CWEEDS2011,CALGARY INTL A,AB,CAN,3031092,51.11,-114.02,-7.00,1084.1                                                     

----------------------------------------------------------------------------------------------------
LINE 3:
ECCC station identifier,File source code (always 'B'),Year Month Day Hour (YYYYMMDDHH),Extraterrestrial irradiance / kJ/m2,Global horizontal irradiance / kJ/m2,Flag,Direct normal irradiance / kJ/m2,Flag,Diffuse horizontal irradiance / kJ/m2,Flag,Global horizontal illuminance / 100 lux,Flag,Direct normal illuminance / 100 lux,Flag,Diffuse horizontal illuminance / 100 lux,Flag,Zenith luminance / 100 Cd/m2,Flag,Minutes of sunshine / 0-60 minutes,Flag,Ceiling height / 10 m,Flag,Sky condition,Flag,Visibility / 100 m,Flag,Present Weath

In [5]:
import csv

with open(files["Calgary"], "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)

    metadata_header = next(reader)
    metadata_values = next(reader)
    weather_header = next(reader)
    first_weather_row = next(reader)

print("Metadata columns :", len(metadata_header))
print("Metadata values  :", len(metadata_values))
print("Weather columns  :", len(weather_header))
print("Weather row      :", len(first_weather_row))

print("\nMetadata:")
for col, value in zip(metadata_header, metadata_values):
    print(f"{col}: {value}")

print("\nLast 10 weather column names:")
print(weather_header[-10:])

print("\nLast 10 values in first weather row:")
print(first_weather_row[-10:])

Metadata columns : 9
Metadata values  : 9
Weather columns  : 44
Weather row      : 45

Metadata:
Data version: CWEEDS2011
Climate station name: CALGARY INTL A
Province: AB
Country: CAN
Station ID: 3031092
Degree latitude: 51.11
Degree longitude: -114.02
UTC offset: -7.00
Elevation: 1084.1                                                     

Last 10 weather column names:
['Wind direction / 0-359 degrees', 'Flag', 'Wind speed / 0.1 m/s', 'Flag', 'Total sky cover / 0-10 in tenths', 'Flag', 'Opaque sky cover / 0-10 in tenths', 'Flag', 'Snow cover (0 = no snow cover 1 = snow cover)', 'Flag']

Last 10 values in first weather row:
['', '78', '', '10', '', '10', '', '0', '', '']


In [6]:
import csv
import pandas as pd


def make_unique_columns(columns):
    """
    Make duplicate column names unique.
    Example:
    Flag, Flag, Flag -> Flag, Flag_2, Flag_3
    """
    counts = {}
    unique_columns = []

    for col in columns:
        col = col.strip()

        if col not in counts:
            counts[col] = 1
            unique_columns.append(col)
        else:
            counts[col] += 1
            unique_columns.append(f"{col}_{counts[col]}")

    return unique_columns


def load_cweeds(file_path):
    """
    Load a CWEEDS CSV file.

    Returns
    -------
    metadata : dict
        Station information from rows 1-2.

    df : pandas.DataFrame
        Hourly weather observations beginning from row 4.
    """

    with open(file_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)

        # Rows 1 and 2: station metadata
        metadata_header = next(reader)
        metadata_values = next(reader)

        metadata = dict(zip(metadata_header, metadata_values))

        # Row 3: weather column names
        weather_header = next(reader)
        weather_header = make_unique_columns(weather_header)

        weather_rows = []

        for row_number, row in enumerate(reader, start=4):

            # Remove extra empty field at end of CWEEDS records
            if (
                len(row) == len(weather_header) + 1
                and row[-1].strip() == ""
            ):
                row = row[:-1]

            # Catch unexpected structural problems
            if len(row) != len(weather_header):
                raise ValueError(
                    f"Row {row_number}: expected "
                    f"{len(weather_header)} fields, "
                    f"found {len(row)}."
                )

            weather_rows.append(row)

    df = pd.DataFrame(weather_rows, columns=weather_header)

    return metadata, df

In [7]:
calgary_metadata, calgary_raw = load_cweeds(files["Calgary"])

print("Calgary loaded successfully!")
print("Rows:", calgary_raw.shape[0])
print("Columns:", calgary_raw.shape[1])

print("\nStation metadata:")
for key, value in calgary_metadata.items():
    print(f"{key}: {value}")

print("\nFirst 3 weather records:")
display(calgary_raw.head(3))

Calgary loaded successfully!
Rows: 175320
Columns: 44

Station metadata:
Data version: CWEEDS2011
Climate station name: CALGARY INTL A
Province: AB
Country: CAN
Station ID: 3031092
Degree latitude: 51.11
Degree longitude: -114.02
UTC offset: -7.00
Elevation: 1084.1                                                     

First 3 weather records:


,ECCC station identifier,File source code (always 'B'),Year Month Day Hour (YYYYMMDDHH),Extraterrestrial irradiance / kJ/m2,Global horizontal irradiance / kJ/m2,Flag,Direct normal irradiance / kJ/m2,Flag_2,Diffuse horizontal irradiance / kJ/m2,Flag_3,Global horizontal illuminance / 100 lux,Flag_4,Direct normal illuminance / 100 lux,Flag_5,Diffuse horizontal illuminance / 100 lux,Flag_6,Zenith luminance / 100 Cd/m2,Flag_7,Minutes of sunshine / 0-60 minutes,Flag_8,Ceiling height / 10 m,Flag_9,Sky condition,Flag_10,Visibility / 100 m,Flag_11,Present Weather,Flag_12,Station pressure / 10 Pa,Flag_13,Dry bulb temperature / 0.1 C,Flag_14,Dew point temperature / 0.1 C,Flag_15,Wind direction / 0-359 degrees,Flag_16,Wind speed / 0.1 m/s,Flag_17,Total sky cover / 0-10 in tenths,Flag_18,Opaque sky cover / 0-10 in tenths,Flag_19,Snow cover (0 = no snow cover 1 = snow cover),Flag_20
0,3031092,A,1998010101,0,0,S,0,S,0,S,0,Q,0,Q,0,Q,9999,9,0,,9,,6000,,48,,00410010,,8786,,-101,,-123,,20,,78,,10,,10,,0,
1,3031092,A,1998010102,0,0,S,0,S,0,S,0,Q,0,Q,0,Q,9999,9,0,,24,,6000,,40,,00010010,,8789,,-116,,-139,,10,,83,,10,,10,,0,
2,3031092,A,1998010103,0,0,S,0,S,0,S,0,Q,0,Q,0,Q,9999,9,0,,27,,6000,,40,,00010010,,8798,,-125,,-148,,10,,61,,10,,10,,0,


In [8]:
city_files = {
    "Calgary": files["Calgary"],
    "Edmonton": files["Edmonton"],
    "Lethbridge": files["Lethbridge"],
    "Medicine Hat": files["Medicine Hat"]
}

city_data = {}
city_metadata = {}
audit_rows = []

for city, file_path in city_files.items():
    metadata, df = load_cweeds(file_path)

    city_data[city] = df
    city_metadata[city] = metadata

    audit_rows.append({
        "City": city,
        "Station Name": metadata["Climate station name"],
        "Station ID": metadata["Station ID"],
        "Latitude": metadata["Degree latitude"],
        "Longitude": metadata["Degree longitude"],
        "Elevation_m": metadata["Elevation"],
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

city_audit = pd.DataFrame(audit_rows)

display(city_audit)

,City,Station Name,Station ID,Latitude,Longitude,Elevation_m,Rows,Columns
0,Calgary,CALGARY INTL A,3031092,51.11,-114.02,1084.1 ...,175320,44
1,Edmonton,EDMONTON INTL A,3012216,53.31,-113.58,723.3 ...,175320,44
2,Lethbridge,LETHBRIDGE CDA,3033890,49.70,-112.77,910.0 ...,175320,44
3,Medicine Hat,MEDICINE HAT RCS,3034485,50.03,-110.72,715.0 ...,175320,44


In [9]:
for city, df in city_data.items():
    timestamp_col = "Year Month Day Hour (YYYYMMDDHH)"

    print(
        f"{city:15} | "
        f"First: {df[timestamp_col].iloc[0]} | "
        f"Last: {df[timestamp_col].iloc[-1]}"
    )

Calgary         | First: 1998010101 | Last: 2017123124
Edmonton        | First: 1998010101 | Last: 2017123124
Lethbridge      | First: 1998010101 | Last: 2017123124
Medicine Hat    | First: 1998010101 | Last: 2017123124


In [10]:
# Inspect all raw CWEEDS column names before renaming

for i, col in enumerate(calgary_raw.columns):
    print(i, "->", col)

0 -> ECCC station identifier
1 -> File source code (always 'B')
2 -> Year Month Day Hour (YYYYMMDDHH)
3 -> Extraterrestrial irradiance / kJ/m2
4 -> Global horizontal irradiance / kJ/m2
5 -> Flag
6 -> Direct normal irradiance / kJ/m2
7 -> Flag_2
8 -> Diffuse horizontal irradiance / kJ/m2
9 -> Flag_3
10 -> Global horizontal illuminance / 100 lux
11 -> Flag_4
12 -> Direct normal illuminance / 100 lux
13 -> Flag_5
14 -> Diffuse horizontal illuminance / 100 lux
15 -> Flag_6
16 -> Zenith luminance / 100 Cd/m2
17 -> Flag_7
18 -> Minutes of sunshine / 0-60 minutes
19 -> Flag_8
20 -> Ceiling height / 10 m
21 -> Flag_9
22 -> Sky condition
23 -> Flag_10
24 -> Visibility / 100 m
25 -> Flag_11
26 -> Present Weather
27 -> Flag_12
28 -> Station pressure / 10 Pa
29 -> Flag_13
30 -> Dry bulb temperature / 0.1 C
31 -> Flag_14
32 -> Dew point temperature / 0.1 C
33 -> Flag_15
34 -> Wind direction / 0-359 degrees
35 -> Flag_16
36 -> Wind speed / 0.1 m/s
37 -> Flag_17
38 -> Total sky cover / 0-10 in tenths

In [11]:
# Standardized names for all 44 CWEEDS columns

CWEEDS_RENAME = {
    "ECCC station identifier": "station_id",
    "File source code (always 'B')": "file_source_code",
    "Year Month Day Hour (YYYYMMDDHH)": "cweeds_timestamp_raw",

    "Extraterrestrial irradiance / kJ/m2": "extraterrestrial_irradiance",
    "Global horizontal irradiance / kJ/m2": "ghi",
    "Flag": "ghi_flag",

    "Direct normal irradiance / kJ/m2": "dni",
    "Flag_2": "dni_flag",

    "Diffuse horizontal irradiance / kJ/m2": "dhi",
    "Flag_3": "dhi_flag",

    "Global horizontal illuminance / 100 lux": "global_horizontal_illuminance",
    "Flag_4": "global_horizontal_illuminance_flag",

    "Direct normal illuminance / 100 lux": "direct_normal_illuminance",
    "Flag_5": "direct_normal_illuminance_flag",

    "Diffuse horizontal illuminance / 100 lux": "diffuse_horizontal_illuminance",
    "Flag_6": "diffuse_horizontal_illuminance_flag",

    "Zenith luminance / 100 Cd/m2": "zenith_luminance",
    "Flag_7": "zenith_luminance_flag",

    "Minutes of sunshine / 0-60 minutes": "sunshine_minutes",
    "Flag_8": "sunshine_flag",

    "Ceiling height / 10 m": "ceiling_height",
    "Flag_9": "ceiling_height_flag",

    "Sky condition": "sky_condition",
    "Flag_10": "sky_condition_flag",

    "Visibility / 100 m": "visibility",
    "Flag_11": "visibility_flag",

    "Present Weather": "present_weather",
    "Flag_12": "present_weather_flag",

    "Station pressure / 10 Pa": "station_pressure",
    "Flag_13": "station_pressure_flag",

    "Dry bulb temperature / 0.1 C": "dry_bulb_temp",
    "Flag_14": "dry_bulb_temp_flag",

    "Dew point temperature / 0.1 C": "dew_point_temp",
    "Flag_15": "dew_point_temp_flag",

    "Wind direction / 0-359 degrees": "wind_direction",
    "Flag_16": "wind_direction_flag",

    "Wind speed / 0.1 m/s": "wind_speed",
    "Flag_17": "wind_speed_flag",

    "Total sky cover / 0-10 in tenths": "total_sky_cover",
    "Flag_18": "total_sky_cover_flag",

    "Opaque sky cover / 0-10 in tenths": "opaque_sky_cover",
    "Flag_19": "opaque_sky_cover_flag",

    "Snow cover (0 = no snow cover 1 = snow cover)": "snow_cover",
    "Flag_20": "snow_cover_flag"
}

In [12]:
def standardize_cweeds(df, city, metadata):
    """
    Standardize CWEEDS column names and create a valid hourly timestamp.
    
    CWEEDS hours are recorded from 01 through 24.
    We convert:
        01 -> 00:00
        02 -> 01:00
        ...
        24 -> 23:00
    """

    df = df.copy()

    # Rename raw CWEEDS columns
    df = df.rename(columns=CWEEDS_RENAME)

    # Keep raw timestamp as text
    df["cweeds_timestamp_raw"] = (
        df["cweeds_timestamp_raw"]
        .astype(str)
        .str.strip()
    )

    # Separate date and CWEEDS hour
    date_part = df["cweeds_timestamp_raw"].str[:8]

    hour_part = pd.to_numeric(
        df["cweeds_timestamp_raw"].str[8:10],
        errors="coerce"
    )

    # Convert YYYYMMDD into date
    base_date = pd.to_datetime(
        date_part,
        format="%Y%m%d",
        errors="coerce"
    )

    # CWEEDS uses hours 01-24
    # Convert these to Python hours 00-23
    df["timestamp"] = (
        base_date +
        pd.to_timedelta(hour_part - 1, unit="h")
    )

    # Add location information
    df.insert(0, "city", city)

    df["station_name"] = metadata["Climate station name"]
    df["latitude"] = float(metadata["Degree latitude"])
    df["longitude"] = float(metadata["Degree longitude"])
    df["elevation_m"] = float(metadata["Elevation"])
    df["utc_offset"] = float(metadata["UTC offset"])

    return df

In [13]:
city_cleaning = {}

for city, df in city_data.items():
    city_cleaning[city] = standardize_cweeds(
        df,
        city,
        city_metadata[city]
    )

print("All four city datasets standardized.")

All four city datasets standardized.


In [14]:
for city, df in city_cleaning.items():

    print(
        f"{city:15} | "
        f"Rows: {len(df):,} | "
        f"First: {df['timestamp'].iloc[0]} | "
        f"Last: {df['timestamp'].iloc[-1]} | "
        f"Invalid timestamps: {df['timestamp'].isna().sum()}"
    )

Calgary         | Rows: 175,320 | First: 1998-01-01 00:00:00 | Last: 2017-12-31 23:00:00 | Invalid timestamps: 0
Edmonton        | Rows: 175,320 | First: 1998-01-01 00:00:00 | Last: 2017-12-31 23:00:00 | Invalid timestamps: 0
Lethbridge      | Rows: 175,320 | First: 1998-01-01 00:00:00 | Last: 2017-12-31 23:00:00 | Invalid timestamps: 0
Medicine Hat    | Rows: 175,320 | First: 1998-01-01 00:00:00 | Last: 2017-12-31 23:00:00 | Invalid timestamps: 0


In [15]:
# Check data types after standardization

calgary_std = city_cleaning["Calgary"]

print(calgary_std.dtypes)

city                                              str
station_id                                        str
file_source_code                                  str
cweeds_timestamp_raw                              str
extraterrestrial_irradiance                       str
ghi                                               str
ghi_flag                                          str
dni                                               str
dni_flag                                          str
dhi                                               str
dhi_flag                                          str
global_horizontal_illuminance                     str
global_horizontal_illuminance_flag                str
direct_normal_illuminance                         str
direct_normal_illuminance_flag                    str
diffuse_horizontal_illuminance                    str
diffuse_horizontal_illuminance_flag               str
zenith_luminance                                  str
zenith_luminance_flag       

In [16]:
# Inspect likely missing-value / sentinel patterns
# We focus first on the variables that matter most for the capstone.

key_columns = [
    "ghi",
    "dni",
    "dhi",
    "sunshine_minutes",
    "station_pressure",
    "dry_bulb_temp",
    "dew_point_temp",
    "wind_direction",
    "wind_speed",
    "total_sky_cover",
    "opaque_sky_cover",
    "snow_cover"
]

for col in key_columns:
    print(f"\n--- {col} ---")
    print("Unique sample:", calgary_std[col].value_counts(dropna=False).head(15))


--- ghi ---
Unique sample: ghi
0     84254
1       510
9       305
2       294
3       294
8       274
10      263
4       259
13      241
16      236
5       230
14      227
7       216
12      214
11      213
Name: count, dtype: int64

--- dni ---
Unique sample: dni
0     99517
4      1034
7       671
11      533
14      325
18      316
1       315
5       313
6       305
2       300
8       293
3       283
15      269
9       231
22      228
Name: count, dtype: int64

--- dhi ---
Unique sample: dhi
0     84392
1       632
4       510
5       497
2       416
7       374
8       358
6       341
3       328
14      321
11      309
12      302
10      292
9       289
13      287
Name: count, dtype: int64

--- sunshine_minutes ---
Unique sample: sunshine_minutes
99    112502
0      37389
60     10098
56       522
4        450
52       410
58       403
57       398
3        360
8        352
10       331
48       316
16       312
50       305
44       304
Name: count, dtype: int64

--- st

In [17]:
flag_columns = [col for col in calgary_std.columns if col.endswith("_flag")]

for col in flag_columns:
    print(f"\n--- {col} ---")
    print(calgary_std[col].value_counts(dropna=False).head(20))


--- ghi_flag ---
ghi_flag
S    174553
M       577
N       126
I        64
Name: count, dtype: int64

--- dni_flag ---
dni_flag
S    174553
Q       641
N       126
Name: count, dtype: int64

--- dhi_flag ---
dhi_flag
S    174553
M       577
N       126
I        64
Name: count, dtype: int64

--- global_horizontal_illuminance_flag ---
global_horizontal_illuminance_flag
Q    175169
9       151
Name: count, dtype: int64

--- direct_normal_illuminance_flag ---
direct_normal_illuminance_flag
Q    169867
9      5453
Name: count, dtype: int64

--- diffuse_horizontal_illuminance_flag ---
diffuse_horizontal_illuminance_flag
Q    175194
9       126
Name: count, dtype: int64

--- zenith_luminance_flag ---
zenith_luminance_flag
9    175320
Name: count, dtype: int64

--- sunshine_flag ---
sunshine_flag
9    112502
      62818
Name: count, dtype: int64

--- ceiling_height_flag ---
ceiling_height_flag
     138646
9     36673
E         1
Name: count, dtype: int64

--- sky_condition_flag ---
sky_conditi

In [18]:
numeric_columns = [
    "extraterrestrial_irradiance",
    "ghi",
    "dni",
    "dhi",
    "global_horizontal_illuminance",
    "direct_normal_illuminance",
    "diffuse_horizontal_illuminance",
    "zenith_luminance",
    "sunshine_minutes",
    "ceiling_height",
    "visibility",
    "station_pressure",
    "dry_bulb_temp",
    "dew_point_temp",
    "wind_direction",
    "wind_speed",
    "total_sky_cover",
    "opaque_sky_cover",
    "snow_cover"
]

city_numeric = {}

for city, df in city_cleaning.items():
    temp = df.copy()

    for col in numeric_columns:
        temp[col] = pd.to_numeric(temp[col], errors="coerce")

    city_numeric[city] = temp

print("Numeric conversion completed for all four cities.")

Numeric conversion completed for all four cities.


In [19]:
conversion_audit = []

for city, df in city_numeric.items():

    for col in numeric_columns:
        conversion_audit.append({
            "City": city,
            "Variable": col,
            "Missing_after_conversion": df[col].isna().sum(),
            "Min": df[col].min(),
            "Max": df[col].max()
        })

conversion_audit = pd.DataFrame(conversion_audit)

display(conversion_audit)

,City,Variable,Missing_after_conversion,Min,Max
0,Calgary,extraterrestrial_irradiance,0,0,4202
1,Calgary,ghi,0,0,3500
2,Calgary,dni,0,0,3682
3,Calgary,dhi,0,0,1916
4,Calgary,global_horizontal_illuminance,0,0,9999
5,Calgary,direct_normal_illuminance,0,0,9999
6,Calgary,diffuse_horizontal_illuminance,0,0,9999
7,Calgary,zenith_luminance,0,9999,9999
8,Calgary,sunshine_minutes,0,0,99
9,Calgary,ceiling_height,0,0,9999


In [20]:
sentinel_candidates = [
    -9999, -999, -99,
    99, 999, 9999,
    777, 7777,
    888, 8888
]

sentinel_results = []

for city, df in city_numeric.items():

    for col in numeric_columns:

        counts = df[col].value_counts()

        for value in sentinel_candidates:

            count = counts.get(value, 0)

            if count > 0:
                sentinel_results.append({
                    "City": city,
                    "Variable": col,
                    "Candidate_Value": value,
                    "Count": count
                })

sentinel_audit = pd.DataFrame(sentinel_results)

display(sentinel_audit)

,City,Variable,Candidate_Value,Count
0,Calgary,extraterrestrial_irradiance,99,30
1,Calgary,extraterrestrial_irradiance,999,40
2,Calgary,extraterrestrial_irradiance,777,25
3,Calgary,ghi,99,57
4,Calgary,ghi,999,36
...,...,...,...,...
155,Medicine Hat,dry_bulb_temp,99,618
156,Medicine Hat,dew_point_temp,-99,341
157,Medicine Hat,dew_point_temp,99,593
158,Medicine Hat,total_sky_cover,99,102803


In [21]:
# Field-specific missing-value codes.
# Only values that are impossible for the defined field are replaced here.

MISSING_CODES = {
    "global_horizontal_illuminance": [9999],
    "direct_normal_illuminance": [9999],
    "diffuse_horizontal_illuminance": [9999],
    "zenith_luminance": [9999],

    "sunshine_minutes": [99],

    "ceiling_height": [9999],
    "visibility": [9999],

    "total_sky_cover": [99],
    "opaque_sky_cover": [99]
}

In [22]:
city_clean = {}

for city, df in city_numeric.items():
    temp = df.copy()

    for col, missing_values in MISSING_CODES.items():
        temp[col] = temp[col].replace(missing_values, np.nan)

    city_clean[city] = temp

print("Field-specific missing codes replaced.")

Field-specific missing codes replaced.


In [23]:
missing_summary = []

for city, df in city_clean.items():
    for col in MISSING_CODES:
        missing_summary.append({
            "City": city,
            "Variable": col,
            "Missing_Count": df[col].isna().sum(),
            "Missing_Percent": round(df[col].isna().mean() * 100, 2)
        })

missing_summary = pd.DataFrame(missing_summary)

display(missing_summary)

,City,Variable,Missing_Count,Missing_Percent
0,Calgary,global_horizontal_illuminance,151,0.09
1,Calgary,direct_normal_illuminance,5453,3.11
2,Calgary,diffuse_horizontal_illuminance,126,0.07
3,Calgary,zenith_luminance,175320,100.00
4,Calgary,sunshine_minutes,112502,64.17
5,Calgary,ceiling_height,36673,20.92
6,Calgary,visibility,137,0.08
7,Calgary,total_sky_cover,32243,18.39
8,Calgary,opaque_sky_cover,48014,27.39
9,Edmonton,global_horizontal_illuminance,138,0.08


In [24]:
MISSING_CODES["extraterrestrial_irradiance"] = [9999]

# Reapply the updated missing-value rules
city_clean = {}

for city, df in city_numeric.items():
    temp = df.copy()

    for col, missing_values in MISSING_CODES.items():
        temp[col] = temp[col].replace(missing_values, np.nan)

    city_clean[city] = temp

print("Updated missing-value handling completed.")

Updated missing-value handling completed.


In [25]:
def add_physical_units(df):
    """
    Add physically interpretable measurement columns
    while preserving original CWEEDS values.
    """

    df = df.copy()

    # Solar radiation
    # CWEEDS values are in kJ/m².
    # 1 Wh = 3.6 kJ.
    df["extraterrestrial_irradiance_wh_m2"] = (
        df["extraterrestrial_irradiance"] / 3.6
    )

    df["ghi_wh_m2"] = df["ghi"] / 3.6
    df["dni_wh_m2"] = df["dni"] / 3.6
    df["dhi_wh_m2"] = df["dhi"] / 3.6

    # Temperature
    df["dry_bulb_temp_c"] = df["dry_bulb_temp"] * 0.1
    df["dew_point_temp_c"] = df["dew_point_temp"] * 0.1

    # Wind
    df["wind_speed_ms"] = df["wind_speed"] * 0.1

    # Pressure
    df["station_pressure_pa"] = df["station_pressure"] * 10

    # Visibility and ceiling
    df["visibility_m"] = df["visibility"] * 100
    df["ceiling_height_m"] = df["ceiling_height"] * 10

    # Illuminance
    df["global_horizontal_illuminance_lux"] = (
        df["global_horizontal_illuminance"] * 100
    )

    df["direct_normal_illuminance_lux"] = (
        df["direct_normal_illuminance"] * 100
    )

    df["diffuse_horizontal_illuminance_lux"] = (
        df["diffuse_horizontal_illuminance"] * 100
    )

    return df

In [26]:
city_scaled = {}

for city, df in city_clean.items():
    city_scaled[city] = add_physical_units(df)

print("Physical unit conversion completed for all four cities.")

Physical unit conversion completed for all four cities.


In [27]:
check_columns = [
    "ghi_wh_m2",
    "dni_wh_m2",
    "dhi_wh_m2",
    "dry_bulb_temp_c",
    "dew_point_temp_c",
    "wind_speed_ms",
    "station_pressure_pa"
]

unit_check = []

for city, df in city_scaled.items():

    for col in check_columns:
        unit_check.append({
            "City": city,
            "Variable": col,
            "Min": round(df[col].min(), 2),
            "Max": round(df[col].max(), 2),
            "Missing": df[col].isna().sum()
        })

unit_check = pd.DataFrame(unit_check)

display(unit_check)

,City,Variable,Min,Max,Missing
0,Calgary,ghi_wh_m2,0.0,972.22,0
1,Calgary,dni_wh_m2,0.0,1022.78,0
2,Calgary,dhi_wh_m2,0.0,532.22,0
3,Calgary,dry_bulb_temp_c,-34.6,34.00,0
4,Calgary,dew_point_temp_c,-39.8,24.00,0
5,Calgary,wind_speed_ms,0.0,21.70,0
6,Calgary,station_pressure_pa,85610.0,91420.00,0
7,Edmonton,ghi_wh_m2,0.0,952.50,0
8,Edmonton,dni_wh_m2,0.0,1010.00,0
9,Edmonton,dhi_wh_m2,0.0,423.89,0


In [28]:
continuity_results = []

for city, df in city_scaled.items():

    temp = df.sort_values("timestamp").copy()

    duplicate_count = temp["timestamp"].duplicated().sum()

    expected_range = pd.date_range(
        start=temp["timestamp"].min(),
        end=temp["timestamp"].max(),
        freq="h"
    )

    actual_timestamps = pd.DatetimeIndex(temp["timestamp"])

    missing_timestamps = expected_range.difference(actual_timestamps)

    continuity_results.append({
        "City": city,
        "Rows": len(temp),
        "Start": temp["timestamp"].min(),
        "End": temp["timestamp"].max(),
        "Expected_Hours": len(expected_range),
        "Duplicate_Timestamps": duplicate_count,
        "Missing_Hours": len(missing_timestamps)
    })

continuity_summary = pd.DataFrame(continuity_results)

display(continuity_summary)

,City,Rows,Start,End,Expected_Hours,Duplicate_Timestamps,Missing_Hours
0,Calgary,175320,1998-01-01,2017-12-31 23:00:00,175320,0,0
1,Edmonton,175320,1998-01-01,2017-12-31 23:00:00,175320,0,0
2,Lethbridge,175320,1998-01-01,2017-12-31 23:00:00,175320,0,0
3,Medicine Hat,175320,1998-01-01,2017-12-31 23:00:00,175320,0,0


In [29]:
# Combine the four standardized and scaled city datasets

master_cweeds = pd.concat(
    city_scaled.values(),
    ignore_index=True
)

print("Master CWEEDS dataset created.")
print("Rows:", f"{len(master_cweeds):,}")
print("Columns:", master_cweeds.shape[1])

display(master_cweeds.head())

Master CWEEDS dataset created.
Rows: 701,280
Columns: 64


,city,station_id,file_source_code,cweeds_timestamp_raw,extraterrestrial_irradiance,ghi,ghi_flag,dni,dni_flag,dhi,dhi_flag,global_horizontal_illuminance,global_horizontal_illuminance_flag,direct_normal_illuminance,direct_normal_illuminance_flag,diffuse_horizontal_illuminance,diffuse_horizontal_illuminance_flag,zenith_luminance,zenith_luminance_flag,sunshine_minutes,sunshine_flag,ceiling_height,ceiling_height_flag,sky_condition,sky_condition_flag,visibility,visibility_flag,present_weather,present_weather_flag,station_pressure,station_pressure_flag,dry_bulb_temp,dry_bulb_temp_flag,dew_point_temp,dew_point_temp_flag,wind_direction,wind_direction_flag,wind_speed,wind_speed_flag,total_sky_cover,total_sky_cover_flag,opaque_sky_cover,opaque_sky_cover_flag,snow_cover,snow_cover_flag,timestamp,station_name,latitude,longitude,elevation_m,utc_offset,extraterrestrial_irradiance_wh_m2,ghi_wh_m2,dni_wh_m2,dhi_wh_m2,dry_bulb_temp_c,dew_point_temp_c,wind_speed_ms,station_pressure_pa,visibility_m,ceiling_height_m,global_horizontal_illuminance_lux,direct_normal_illuminance_lux,diffuse_horizontal_illuminance_lux
0,Calgary,3031092,A,1998010101,0.0,0,S,0,S,0,S,0.0,Q,0.0,Q,0.0,Q,NaN,9,0.0,,9.0,,6000,,48.0,,00410010,,8786,,-101,,-123,,20,,78,,10.0,,10.0,,0,,1998-01-01 00:00:00,CALGARY INTL A,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0,0.0,-10.1,-12.3,7.8,87860,4800.0,90.0,0.0,0.0,0.0
1,Calgary,3031092,A,1998010102,0.0,0,S,0,S,0,S,0.0,Q,0.0,Q,0.0,Q,NaN,9,0.0,,24.0,,6000,,40.0,,00010010,,8789,,-116,,-139,,10,,83,,10.0,,10.0,,0,,1998-01-01 01:00:00,CALGARY INTL A,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0,0.0,-11.6,-13.9,8.3,87890,4000.0,240.0,0.0,0.0,0.0
2,Calgary,3031092,A,1998010103,0.0,0,S,0,S,0,S,0.0,Q,0.0,Q,0.0,Q,NaN,9,0.0,,27.0,,6000,,40.0,,00010010,,8798,,-125,,-148,,10,,61,,10.0,,10.0,,0,,1998-01-01 02:00:00,CALGARY INTL A,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0,0.0,-12.5,-14.8,6.1,87980,4000.0,270.0,0.0,0.0,0.0
3,Calgary,3031092,A,1998010104,0.0,0,S,0,S,0,S,0.0,Q,0.0,Q,0.0,Q,NaN,9,0.0,,27.0,,6000,,80.0,,00010000,,8795,,-134,,-156,,0,,67,,10.0,,10.0,,0,,1998-01-01 03:00:00,CALGARY INTL A,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0,0.0,-13.4,-15.6,6.7,87950,8000.0,270.0,0.0,0.0,0.0
4,Calgary,3031092,A,1998010105,0.0,0,S,0,S,0,S,0.0,Q,0.0,Q,0.0,Q,NaN,9,0.0,,57.0,,2600,,161.0,,00010000,,8798,,-147,,-173,,350,,67,,10.0,,10.0,,0,,1998-01-01 04:00:00,CALGARY INTL A,51.11,-114.02,1084.1,-7.0,0.0,0.0,0.0,0.0,-14.7,-17.3,6.7,87980,16100.0,570.0,0.0,0.0,0.0


In [30]:
city_counts = (
    master_cweeds["city"]
    .value_counts()
    .rename_axis("City")
    .reset_index(name="Rows")
)

display(city_counts)

,City,Rows
0,Calgary,175320
1,Edmonton,175320
2,Lethbridge,175320
3,Medicine Hat,175320


In [31]:
master_summary = (
    master_cweeds
    .groupby("city")
    .agg(
        rows=("timestamp", "size"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
        station=("station_name", "first"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        elevation_m=("elevation_m", "first")
    )
    .reset_index()
)

display(master_summary)

,city,rows,start,end,station,latitude,longitude,elevation_m
0,Calgary,175320,1998-01-01,2017-12-31 23:00:00,CALGARY INTL A,51.11,-114.02,1084.1
1,Edmonton,175320,1998-01-01,2017-12-31 23:00:00,EDMONTON INTL A,53.31,-113.58,723.3
2,Lethbridge,175320,1998-01-01,2017-12-31 23:00:00,LETHBRIDGE CDA,49.70,-112.77,910.0
3,Medicine Hat,175320,1998-01-01,2017-12-31 23:00:00,MEDICINE HAT RCS,50.03,-110.72,715.0


In [32]:
# Check missingness for the core analysis variables

core_analysis_columns = [
    "ghi_wh_m2",
    "dni_wh_m2",
    "dhi_wh_m2",
    "dry_bulb_temp_c",
    "dew_point_temp_c",
    "wind_speed_ms",
    "wind_direction",
    "station_pressure_pa"
]

core_quality = []

for city, group in master_cweeds.groupby("city"):
    for col in core_analysis_columns:
        core_quality.append({
            "City": city,
            "Variable": col,
            "Missing_Count": group[col].isna().sum(),
            "Missing_Percent": round(group[col].isna().mean() * 100, 4),
            "Min": group[col].min(),
            "Max": group[col].max()
        })

core_quality_summary = pd.DataFrame(core_quality)

display(core_quality_summary)

,City,Variable,Missing_Count,Missing_Percent,Min,Max
0,Calgary,ghi_wh_m2,0,0.0,0.0,972.222222
1,Calgary,dni_wh_m2,0,0.0,0.0,1022.777778
2,Calgary,dhi_wh_m2,0,0.0,0.0,532.222222
3,Calgary,dry_bulb_temp_c,0,0.0,-34.6,34.000000
4,Calgary,dew_point_temp_c,0,0.0,-39.8,24.000000
5,Calgary,wind_speed_ms,0,0.0,0.0,21.700000
6,Calgary,wind_direction,0,0.0,0.0,350.000000
7,Calgary,station_pressure_pa,0,0.0,85610.0,91420.000000
8,Edmonton,ghi_wh_m2,0,0.0,0.0,952.500000
9,Edmonton,dni_wh_m2,0,0.0,0.0,1010.000000


In [33]:
quality_checks = []

for city, group in master_cweeds.groupby("city"):

    quality_checks.append({
        "City": city,
        "Negative_GHI": (group["ghi_wh_m2"] < 0).sum(),
        "Negative_DNI": (group["dni_wh_m2"] < 0).sum(),
        "Negative_DHI": (group["dhi_wh_m2"] < 0).sum(),
        "Sunshine_Over_60": (group["sunshine_minutes"] > 60).sum(),
        "Sky_Cover_Over_10": (group["total_sky_cover"] > 10).sum(),
        "Wind_Direction_Over_360": (group["wind_direction"] > 360).sum(),
        "Snow_Cover_Invalid": (~group["snow_cover"].isin([0, 1])).sum()
    })

quality_check_summary = pd.DataFrame(quality_checks)

display(quality_check_summary)

,City,Negative_GHI,Negative_DNI,Negative_DHI,Sunshine_Over_60,Sky_Cover_Over_10,Wind_Direction_Over_360,Snow_Cover_Invalid
0,Calgary,0,0,0,0,0,0,0
1,Edmonton,0,0,0,0,0,0,0
2,Lethbridge,0,0,0,0,0,0,0
3,Medicine Hat,0,0,0,0,0,0,0


In [34]:
from pathlib import Path

CLEAN_DIR = DATA_DIR / "cleaned"
CLEAN_DIR.mkdir(exist_ok=True)

master_cweeds.to_csv(
    CLEAN_DIR / "clean_cweeds_4cities.csv",
    index=False
)

print("Saved to:")
print(CLEAN_DIR / "clean_cweeds_4cities.csv")

Saved to:
C:\Users\aduak\Downloads\capstone\cleaned\clean_cweeds_4cities.csv


In [35]:
city_audit.to_csv(
    CLEAN_DIR / "cweeds_station_audit.csv",
    index=False
)

missing_summary.to_csv(
    CLEAN_DIR / "cweeds_missingness_summary.csv",
    index=False
)

continuity_summary.to_csv(
    CLEAN_DIR / "cweeds_continuity_summary.csv",
    index=False
)

quality_check_summary.to_csv(
    CLEAN_DIR / "cweeds_physical_quality_checks.csv",
    index=False
)

print("Quality reports saved.")

Quality reports saved.


In [36]:
sait_raw = pd.read_csv(files["SAIT"])

print("SAIT dataset shape:", sait_raw.shape)

display(sait_raw.head())

print("\nColumn types:")
print(sait_raw.dtypes)

print("\nColumns:")
for i, col in enumerate(sait_raw.columns):
    print(i, "->", col)

SAIT dataset shape: (1048575, 4)


,ts,Carport Solar,Rooftop Solar,GBT Consumption Hourly Wh
0,4/8/2018 2:15,NaN,NaN,NaN
1,4/8/2018 2:30,NaN,NaN,NaN
2,4/8/2018 2:45,NaN,NaN,NaN
3,4/8/2018 3:00,NaN,NaN,NaN
4,4/8/2018 3:15,NaN,NaN,NaN



Column types:
ts                               str
Carport Solar                float64
Rooftop Solar                float64
GBT Consumption Hourly Wh    float64
dtype: object

Columns:
0 -> ts
1 -> Carport Solar
2 -> Rooftop Solar
3 -> GBT Consumption Hourly Wh


In [37]:
sait = sait_raw.copy()

# Convert timestamp text to datetime
sait["timestamp"] = pd.to_datetime(
    sait["ts"],
    errors="coerce"
)

print("Invalid timestamps:", sait["timestamp"].isna().sum())
print("Start:", sait["timestamp"].min())
print("End:", sait["timestamp"].max())

Invalid timestamps: 0
Start: 2018-04-08 02:15:00
End: 2023-09-19 09:57:00


In [38]:
# Basic missingness audit

sait_missing = pd.DataFrame({
    "Variable": [
        "Carport Solar",
        "Rooftop Solar",
        "GBT Consumption Hourly Wh"
    ],
    "Missing_Count": [
        sait["Carport Solar"].isna().sum(),
        sait["Rooftop Solar"].isna().sum(),
        sait["GBT Consumption Hourly Wh"].isna().sum()
    ]
})

sait_missing["Missing_Percent"] = (
    sait_missing["Missing_Count"] / len(sait) * 100
).round(2)

display(sait_missing)

,Variable,Missing_Count,Missing_Percent
0,Carport Solar,241850,23.06
1,Rooftop Solar,241855,23.07
2,GBT Consumption Hourly Wh,1034382,98.65


In [39]:
sait_sorted = sait.sort_values("timestamp").copy()

sait_sorted["time_gap_minutes"] = (
    sait_sorted["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

gap_counts = (
    sait_sorted["time_gap_minutes"]
    .value_counts()
    .sort_index()
    .head(30)
)

display(gap_counts)

time_gap_minutes
1.0      1024198
2.0          693
3.0          532
4.0          301
5.0          464
6.0          299
7.0          453
8.0          234
9.0          248
10.0         602
11.0           2
12.0          29
13.0           1
14.0          13
15.0       16146
20.0           1
23.0           2
30.0        1673
31.0           1
45.0           2
46.0           1
55.0           1
56.0           1
60.0        2615
61.0           3
68.0           1
90.0           2
92.0           1
98.0           1
110.0          1
Name: count, dtype: int64

In [40]:
# Add year and date helpers

sait_sorted["year"] = sait_sorted["timestamp"].dt.year
sait_sorted["date"] = sait_sorted["timestamp"].dt.date

# Show the most common time gaps by year

gap_by_year = (
    sait_sorted
    .groupby(["year", "time_gap_minutes"])
    .size()
    .reset_index(name="count")
)

# Keep the top 8 gap types per year
gap_by_year["rank"] = (
    gap_by_year
    .groupby("year")["count"]
    .rank(method="first", ascending=False)
)

display(
    gap_by_year[gap_by_year["rank"] <= 8]
    .sort_values(["year", "count"], ascending=[True, False])
)

,year,time_gap_minutes,count,rank
0,2018,1.0,8394,1.0
8,2018,30.0,1032,2.0
7,2018,15.0,972,3.0
6,2018,12.0,29,4.0
2,2018,3.0,27,5.0
5,2018,9.0,6,6.0
3,2018,4.0,4,7.0
4,2018,5.0,4,8.0
32,2019,1.0,307432,1.0
39,2019,15.0,1229,2.0


In [41]:
# Check measurement completeness by year

yearly_completeness = (
    sait_sorted
    .groupby("year")
    .agg(
        rows=("timestamp", "size"),
        carport_valid=("Carport Solar", "count"),
        rooftop_valid=("Rooftop Solar", "count"),
        consumption_valid=("GBT Consumption Hourly Wh", "count")
    )
    .reset_index()
)

yearly_completeness["carport_percent"] = (
    yearly_completeness["carport_valid"] /
    yearly_completeness["rows"] * 100
).round(2)

yearly_completeness["rooftop_percent"] = (
    yearly_completeness["rooftop_valid"] /
    yearly_completeness["rows"] * 100
).round(2)

yearly_completeness["consumption_percent"] = (
    yearly_completeness["consumption_valid"] /
    yearly_completeness["rows"] * 100
).round(2)

display(yearly_completeness)

,year,rows,carport_valid,rooftop_valid,consumption_valid,carport_percent,rooftop_percent,consumption_percent
0,2018,10494,0,0,0,0.00,0.00,0.00
1,2019,308801,99195,99190,1617,32.12,32.12,0.52
2,2020,211373,196071,196070,3856,92.76,92.76,1.82
3,2021,62168,55809,55809,1127,89.77,89.77,1.81
4,2022,85603,85603,85603,1426,100.00,100.00,1.67
5,2023,370136,370047,370048,6167,99.98,99.98,1.67


In [42]:
year_coverage = (
    sait_sorted
    .groupby("year")
    .agg(
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
        rows=("timestamp", "size"),
        unique_days=("date", "nunique"),
        carport_valid=("Carport Solar", "count"),
        rooftop_valid=("Rooftop Solar", "count")
    )
    .reset_index()
)

display(year_coverage)

,year,first_timestamp,last_timestamp,rows,unique_days,carport_valid,rooftop_valid
0,2018,2018-04-08 02:15:00,2018-12-31 22:45:00,10494,50,0,0
1,2019,2019-01-01 07:26:00,2019-12-31 23:45:00,308801,252,99195,99190
2,2020,2020-01-01 00:00:00,2020-12-18 17:30:00,211373,334,196071,196070
3,2021,2021-01-07 18:00:00,2021-06-29 15:24:00,62168,124,55809,55809
4,2022,2022-11-02 13:17:00,2022-12-31 23:59:00,85603,60,85603,85603
5,2023,2023-01-01 00:00:00,2023-09-19 09:57:00,370136,262,370047,370048


In [43]:
largest_gaps = (
    sait_sorted[
        [
            "timestamp",
            "time_gap_minutes"
        ]
    ]
    .sort_values(
        "time_gap_minutes",
        ascending=False
    )
    .head(20)
)

display(largest_gaps)

,timestamp,time_gap_minutes
592836,2022-11-02 13:17:00,706913.0
1444,2018-10-13 15:15:00,234585.0
589974,2021-05-19 00:00:00,73341.0
318165,2019-12-20 05:45:00,35625.0
530668,2021-01-07 18:00:00,28830.0
4195,2018-12-08 02:15:00,28725.0
466841,2020-07-24 22:00:00,22265.0
317956,2019-10-13 06:00:00,19464.0
10552,2019-02-13 07:05:00,17526.0
317970,2019-11-04 02:45:00,16845.0


In [44]:
sait_sorted["previous_timestamp"] = (
    sait_sorted["timestamp"].shift(1)
)

largest_gap_details = (
    sait_sorted[
        [
            "previous_timestamp",
            "timestamp",
            "time_gap_minutes"
        ]
    ]
    .sort_values(
        "time_gap_minutes",
        ascending=False
    )
    .head(20)
)

display(largest_gap_details)

,previous_timestamp,timestamp,time_gap_minutes
592836,2021-06-29 15:24:00,2022-11-02 13:17:00,706913.0
1444,2018-05-03 17:30:00,2018-10-13 15:15:00,234585.0
589974,2021-03-29 01:39:00,2021-05-19 00:00:00,73341.0
318165,2019-11-25 12:00:00,2019-12-20 05:45:00,35625.0
530668,2020-12-18 17:30:00,2021-01-07 18:00:00,28830.0
4195,2018-11-18 03:30:00,2018-12-08 02:15:00,28725.0
466841,2020-07-09 10:55:00,2020-07-24 22:00:00,22265.0
317956,2019-09-29 17:36:00,2019-10-13 06:00:00,19464.0
10552,2019-02-01 02:59:00,2019-02-13 07:05:00,17526.0
317970,2019-10-23 10:00:00,2019-11-04 02:45:00,16845.0


In [45]:
negative_summary = []

for col in ["Carport Solar", "Rooftop Solar"]:
    series = sait_sorted[col]

    negative_summary.append({
        "Variable": col,
        "Negative_Count": (series < 0).sum(),
        "Negative_Percent_of_Valid": round(
            ((series < 0).sum() / series.notna().sum()) * 100, 4
        ),
        "Min_Value": series.min(),
        "Max_Value": series.max()
    })

negative_summary = pd.DataFrame(negative_summary)

display(negative_summary)

,Variable,Negative_Count,Negative_Percent_of_Valid,Min_Value,Max_Value
0,Carport Solar,71293,8.8373,-88.0,15391.0
1,Rooftop Solar,27,0.0033,-2.0,7199.0


In [46]:
negative_rows = sait_sorted[
    (sait_sorted["Carport Solar"] < 0) |
    (sait_sorted["Rooftop Solar"] < 0)
].copy()

negative_rows["hour"] = negative_rows["timestamp"].dt.hour

negative_by_hour = (
    negative_rows
    .groupby("hour")
    .agg(
        negative_rows=("timestamp", "size"),
        carport_min=("Carport Solar", "min"),
        rooftop_min=("Rooftop Solar", "min")
    )
    .reset_index()
)

display(negative_by_hour)

,hour,negative_rows,carport_min,rooftop_min
0,0,5928,-28.0,-1.0
1,1,5883,-27.0,-1.0
2,2,5843,-28.0,0.0
3,3,5821,-27.0,-1.0
4,4,5785,-37.0,-1.0
5,5,5352,-48.0,-1.0
6,6,3157,-42.0,-1.0
7,7,2765,-50.0,-1.0
8,8,2013,-88.0,-1.0
9,9,700,-19.0,1.0


In [47]:
display(
    negative_rows[
        [
            "timestamp",
            "Carport Solar",
            "Rooftop Solar"
        ]
    ].head(30)
)

,timestamp,Carport Solar,Rooftop Solar
22948,2019-03-08 19:35:00,-7.0,NaN
22949,2019-03-08 19:36:00,-9.0,NaN
22950,2019-03-08 19:37:00,-9.0,NaN
22951,2019-03-08 19:38:00,-8.0,NaN
22952,2019-03-08 19:39:00,-9.0,NaN
22953,2019-03-08 19:40:00,-8.0,2.0
22954,2019-03-08 19:41:00,-8.0,2.0
22955,2019-03-08 19:42:00,-7.0,3.0
22956,2019-03-08 19:43:00,-8.0,3.0
22957,2019-03-08 19:44:00,-5.0,4.0


In [48]:
sait_clean = sait_sorted.copy()

# Preserve original values
sait_clean["carport_solar_raw"] = sait_clean["Carport Solar"]
sait_clean["rooftop_solar_raw"] = sait_clean["Rooftop Solar"]

# Clean negative generation readings
sait_clean["carport_solar_clean"] = (
    sait_clean["Carport Solar"].clip(lower=0)
)

sait_clean["rooftop_solar_clean"] = (
    sait_clean["Rooftop Solar"].clip(lower=0)
)

# Flag rows where a negative value was corrected
sait_clean["carport_negative_corrected"] = (
    sait_clean["Carport Solar"] < 0
)

sait_clean["rooftop_negative_corrected"] = (
    sait_clean["Rooftop Solar"] < 0
)

print("Negative solar readings clipped to zero.")

Negative solar readings clipped to zero.


In [49]:
cleaning_check = pd.DataFrame({
    "Variable": [
        "Carport Solar",
        "Rooftop Solar"
    ],
    "Original_Negatives": [
        (sait_clean["carport_solar_raw"] < 0).sum(),
        (sait_clean["rooftop_solar_raw"] < 0).sum()
    ],
    "Cleaned_Negatives": [
        (sait_clean["carport_solar_clean"] < 0).sum(),
        (sait_clean["rooftop_solar_clean"] < 0).sum()
    ],
    "Corrected_Rows": [
        sait_clean["carport_negative_corrected"].sum(),
        sait_clean["rooftop_negative_corrected"].sum()
    ]
})

display(cleaning_check)

,Variable,Original_Negatives,Cleaned_Negatives,Corrected_Rows
0,Carport Solar,71293,0,71293
1,Rooftop Solar,27,0,27


In [50]:
sait_clean["total_solar_clean"] = (
    sait_clean["carport_solar_clean"] +
    sait_clean["rooftop_solar_clean"]
)

In [51]:
sait_hourly = (
    sait_clean
    .set_index("timestamp")
    .resample("h")
    .agg({
        "carport_solar_clean": ["sum", "count"],
        "rooftop_solar_clean": ["sum", "count"],
        "total_solar_clean": ["sum", "count"]
    })
)

# Flatten column names
sait_hourly.columns = [
    "carport_solar_sum",
    "carport_observation_count",
    "rooftop_solar_sum",
    "rooftop_observation_count",
    "total_solar_sum",
    "total_observation_count"
]

sait_hourly = sait_hourly.reset_index()

display(sait_hourly.head())

,timestamp,carport_solar_sum,carport_observation_count,rooftop_solar_sum,rooftop_observation_count,total_solar_sum,total_observation_count
0,2018-04-08 02:00:00,0.0,0,0.0,0,0.0,0
1,2018-04-08 03:00:00,0.0,0,0.0,0,0.0,0
2,2018-04-08 04:00:00,0.0,0,0.0,0,0.0,0
3,2018-04-08 05:00:00,0.0,0,0.0,0,0.0,0
4,2018-04-08 06:00:00,0.0,0,0.0,0,0.0,0


In [52]:
no_carport_data = (
    sait_hourly["carport_observation_count"] == 0
)

no_rooftop_data = (
    sait_hourly["rooftop_observation_count"] == 0
)

no_total_data = (
    sait_hourly["total_observation_count"] == 0
)

sait_hourly.loc[
    no_carport_data,
    "carport_solar_sum"
] = np.nan

sait_hourly.loc[
    no_rooftop_data,
    "rooftop_solar_sum"
] = np.nan

sait_hourly.loc[
    no_total_data,
    "total_solar_sum"
] = np.nan

In [53]:
sait_hourly["year"] = sait_hourly["timestamp"].dt.year
sait_hourly["month"] = sait_hourly["timestamp"].dt.month
sait_hourly["day"] = sait_hourly["timestamp"].dt.day
sait_hourly["hour"] = sait_hourly["timestamp"].dt.hour

In [54]:
display(
    sait_hourly[
        [
            "carport_observation_count",
            "rooftop_observation_count",
            "total_observation_count"
        ]
    ]
    .describe()
)

,carport_observation_count,rooftop_observation_count,total_observation_count
count,47768.000000,47768.000000,47768.000000
mean,16.888398,16.888293,16.887938
std,26.976061,26.975808,26.975840
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,60.000000,60.000000,60.000000
max,60.000000,60.000000,60.000000


In [55]:
hourly_quality = pd.DataFrame({
    "Metric": [
        "Total hourly rows",
        "Hours with no Carport data",
        "Hours with no Rooftop data",
        "Hours with no solar data",
        "Hours with >= 60 solar observations",
        "Hours with 1-59 solar observations"
    ],
    "Count": [
        len(sait_hourly),
        (sait_hourly["carport_observation_count"] == 0).sum(),
        (sait_hourly["rooftop_observation_count"] == 0).sum(),
        (sait_hourly["total_observation_count"] == 0).sum(),
        (sait_hourly["total_observation_count"] >= 60).sum(),
        (
            (sait_hourly["total_observation_count"] > 0) &
            (sait_hourly["total_observation_count"] < 60)
        ).sum()
    ]
})

display(hourly_quality)

,Metric,Count
0,Total hourly rows,47768
1,Hours with no Carport data,34304
2,Hours with no Rooftop data,34304
3,Hours with no solar data,34306
4,Hours with >= 60 solar observations,13423
5,Hours with 1-59 solar observations,39


In [56]:
# Find the full hour with the highest average Carport Solar reading

sait_peak_check = (
    sait_clean
    .set_index("timestamp")
    .resample("h")
    .agg(
        carport_count=("carport_solar_clean", "count"),
        carport_mean=("carport_solar_clean", "mean"),
        carport_max=("carport_solar_clean", "max"),
        carport_sum=("carport_solar_clean", "sum")
    )
)

display(
    sait_peak_check[
        sait_peak_check["carport_count"] == 60
    ]
    .sort_values("carport_mean", ascending=False)
    .head(10)
)

,carport_count,carport_mean,carport_max,carport_sum
timestamp,,,,
2023-06-25 12:00:00,60,14141.833333,15077.0,848510.0
2020-06-15 12:00:00,60,13838.350000,14551.0,830301.0
2020-05-28 13:00:00,60,13755.200000,14368.0,825312.0
2020-06-03 14:00:00,60,13711.366667,14824.0,822682.0
2020-04-26 12:00:00,60,13669.883333,15060.0,820193.0
2023-05-14 13:00:00,60,13582.433333,13771.0,814946.0
2020-06-03 12:00:00,60,13550.433333,15025.0,813026.0
2020-05-28 12:00:00,60,13532.083333,13984.0,811925.0
2020-04-28 13:00:00,60,13471.850000,13685.0,808311.0


In [61]:
sait_hourly = (
    sait_clean
    .set_index("timestamp")
    .resample("h")
    .agg(
        carport_hourly_wh=("carport_solar_clean", "mean"),
        carport_observations=("carport_solar_clean", "count"),

        rooftop_hourly_wh=("rooftop_solar_clean", "mean"),
        rooftop_observations=("rooftop_solar_clean", "count"),

        total_hourly_wh=("total_solar_clean", "mean"),
        total_observations=("total_solar_clean", "count")
    )
    .reset_index()
)

print("SAIT hourly dataset created.")

SAIT hourly dataset created.


In [58]:
def classify_hour(count):
    if count == 60:
        return "complete"
    elif count > 0:
        return "partial"
    else:
        return "missing"


sait_hourly["quality_flag"] = (
    sait_hourly["total_observations"]
    .apply(classify_hour)
)

In [59]:
display(
    sait_hourly["quality_flag"]
    .value_counts()
    .rename_axis("Quality")
    .reset_index(name="Hours")
)

,Quality,Hours
0,missing,34306
1,complete,13423
2,partial,39


In [60]:
sait_hourly["carport_hourly_wh"] = sait_hourly["carport_avg"]
sait_hourly["rooftop_hourly_wh"] = sait_hourly["rooftop_avg"]
sait_hourly["total_hourly_wh"] = sait_hourly["total_solar_avg"]

In [62]:
def classify_hour(count):
    if count == 60:
        return "complete"
    elif count > 0:
        return "partial"
    else:
        return "missing"


sait_hourly["quality_flag"] = (
    sait_hourly["total_observations"]
    .apply(classify_hour)
)

In [63]:
sait_hourly["year"] = sait_hourly["timestamp"].dt.year
sait_hourly["month"] = sait_hourly["timestamp"].dt.month
sait_hourly["day"] = sait_hourly["timestamp"].dt.day
sait_hourly["hour"] = sait_hourly["timestamp"].dt.hour

In [64]:
display(
    sait_hourly["quality_flag"]
    .value_counts()
    .rename_axis("Quality")
    .reset_index(name="Hours")
)

,Quality,Hours
0,missing,34306
1,complete,13423
2,partial,39


In [65]:
sait_hourly_valid = (
    sait_hourly[
        sait_hourly["quality_flag"] == "complete"
    ]
    .copy()
)

print("High-quality hourly records:", len(sait_hourly_valid))

High-quality hourly records: 13423


In [66]:
valid_by_year = (
    sait_hourly_valid
    .groupby("year")
    .agg(
        valid_hours=("timestamp", "size"),
        first_valid=("timestamp", "min"),
        last_valid=("timestamp", "max")
    )
    .reset_index()
)

display(valid_by_year)

,year,valid_hours,first_valid,last_valid
0,2019,1650,2019-03-08 20:00:00,2019-09-29 16:00:00
1,2020,3256,2020-01-10 13:00:00,2020-12-01 23:00:00
2,2021,928,2021-02-18 09:00:00,2021-06-29 14:00:00
3,2022,1426,2022-11-02 14:00:00,2022-12-31 23:00:00
4,2023,6163,2023-01-01 00:00:00,2023-09-19 08:00:00


In [67]:
valid_by_month = (
    sait_hourly_valid
    .groupby(["year", "month"])
    .size()
    .reset_index(name="valid_hours")
)

display(valid_by_month)

,year,month,valid_hours
0,2019,3,34
1,2019,7,183
2,2019,8,744
3,2019,9,689
4,2020,1,515
5,2020,2,472
6,2020,3,426
7,2020,4,178
8,2020,5,373
9,2020,6,355


In [68]:
# Save cleaned SAIT datasets

sait_clean.to_csv(
    CLEAN_DIR / "clean_sait_solar.csv",
    index=False
)

sait_hourly.to_csv(
    CLEAN_DIR / "sait_hourly_all.csv",
    index=False
)

sait_hourly_valid.to_csv(
    CLEAN_DIR / "sait_hourly_valid.csv",
    index=False
)

valid_by_year.to_csv(
    CLEAN_DIR / "sait_valid_hours_by_year.csv",
    index=False
)

valid_by_month.to_csv(
    CLEAN_DIR / "sait_valid_hours_by_month.csv",
    index=False
)

print("SAIT cleaned datasets and quality summaries saved.")

SAIT cleaned datasets and quality summaries saved.
